# Question 1 - 88. Merge Sorted Array

## Description

You are given two integer arrays `nums1` and `nums2`, sorted in non-decreasing order, and two integers `m` and `n`, representing the number of elements in `nums1` and `nums2` respectively.

Merge `nums1` and `nums2` into a single array sorted in non-decreasing order.

The final sorted array should not be returned by the function, but instead be stored inside the array `nums1`. To accommodate this, `nums1` has a length of `m + n`, where the first `m` elements denote the elements that should be merged, and the last `n` elements are set to 0 and should be ignored. `nums2` has a length of `n`.

### Example 1:

- **Input:** `nums1 = [1,2,3,0,0,0]`, `m = 3`, `nums2 = [2,5,6]`, `n = 3`
- **Output:** `[1,2,2,3,5,6]`
- **Explanation:** The arrays we are merging are `[1,2,3]` and `[2,5,6]`.
  The result of the merge is `[1,2,2,3,5,6]` with the underlined elements coming from `nums1`.

### Example 2:

- **Input:** `nums1 = [1]`, `m = 1`, `nums2 = []`, `n = 0`
- **Output:** `[1]`
- **Explanation:** The arrays we are merging are `[1]` and `[]`.
  The result of the merge is `[1]`.

### Example 3:

- **Input:** `nums1 = [0]`, `m = 0`, `nums2 = [1]`, `n = 1`
- **Output:** `[1]`
- **Explanation:** The arrays we are merging are `[]` and `[1]`.
  The result of the merge is `[1]`.
  Note that because `m = 0`, there are no elements in `nums1`. The 0 is only there to ensure the merge result can fit in `nums1`.

### Constraints:

- `nums1.length == m + n`
- `nums2.length == n`
- `0 <= m, n <= 200`
- `1 <= m + n <= 200`
- `-10^9 <= nums1[i], nums2[j] <= 10^9`

### Follow up:

Can you come up with an algorithm that runs in O(m + n) time?


---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Two pointers (fill from the back)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force | O((m+n) log(m+n)) | O(1)* |
| 2️⃣ Optimized | O(m+n) | O(1) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding


🗣️ *"Let me make sure I have the contract right before I code."*

- Is `nums1` guaranteed to have exactly `n` spare slots at the end? → **Yes**, `len(nums1) == m + n`.
- Do I return anything? → **No**, modify `nums1` in place.
- Can values be negative or duplicated? → **Yes**, both.
- Edge cases I'll keep in mind: `n == 0` (nothing to do), `m == 0` (copy all of `nums2`).

### Step 1 · Brute force


**Idea:** Ignore the fact that both arrays are already sorted. Copy `nums2` into the empty tail of `nums1`, then sort the whole thing.

🗣️ *"The simplest correct thing I can do is drop `nums2` into the zero slots and call sort. That's O((m+n) log(m+n)). It's correct, but it throws away the information that both halves are already sorted, so I think we can do better."*

\* Python's Timsort can use up to O(m+n) extra memory internally.

In [1]:
class SolutionBrute:
    def merge(self, nums1: list[int], m: int, nums2: list[int], n: int) -> None:
        nums1[m:] = nums2   # put nums2 into the spare slots
        nums1.sort()        # let the library do the work

### Step 2 · Optimize


**Bottleneck:** Sorting re-discovers order we already have.

**Key insight:** A normal merge (smallest first, from the front) would overwrite `nums1` values we haven't read yet. But the **end** of `nums1` is empty. If we merge from the back, placing the **largest** remaining element at index `k`, we only ever write into slots that are free or already consumed.

**Algorithm:**
1. `i = m-1` (last real value of `nums1`), `j = n-1` (last of `nums2`), `k = m+n-1` (write position).
2. While `j >= 0`: if `i >= 0` and `nums1[i] > nums2[j]`, write `nums1[i]` and move `i`; otherwise write `nums2[j]` and move `j`. Then move `k`.
3. Stop when `nums2` is exhausted. Whatever is left of `nums1` is already in place.

🗣️ *"Since both arrays are sorted, I can merge in linear time. The trick is to fill from the back: the tail of `nums1` is free space, so writing the largest element there never clobbers anything I still need. I'll loop on `j`, because once `nums2` is empty, the rest of `nums1` is already correct."*

In [2]:
class Solution:
    def merge(self, nums1: list[int], m: int, nums2: list[int], n: int) -> None:
        i, j, k = m - 1, n - 1, m + n - 1
        while j >= 0:                          # only nums2 can still be out of place
            if i >= 0 and nums1[i] > nums2[j]:
                nums1[k] = nums1[i]
                i -= 1
            else:
                nums1[k] = nums2[j]
                j -= 1
            k -= 1

### Step 3 · Toy example walkthrough


`nums1 = [1, 2, 3, 0, 0, 0]`, `m = 3`, `nums2 = [2, 5, 6]`, `n = 3`

| step | i → nums1[i] | j → nums2[j] | compare | write at k | nums1 after |
|---|---|---|---|---|---|
| 1 | 2 → 3 | 2 → 6 | 3 > 6? no | k=5 ← 6 | [1,2,3,0,0,**6**] |
| 2 | 2 → 3 | 1 → 5 | 3 > 5? no | k=4 ← 5 | [1,2,3,0,**5**,6] |
| 3 | 2 → 3 | 0 → 2 | 3 > 2? **yes** | k=3 ← 3 | [1,2,3,**3**,5,6] |
| 4 | 1 → 2 | 0 → 2 | 2 > 2? no (tie → take nums2) | k=2 ← 2 | [1,2,**2**,3,5,6] |
| – | 1 | -1 | j < 0 → stop | | **[1,2,2,3,5,6]** ✅ |

Notice we never read a slot after overwriting it: `k` always stays ≥ `i`.

In [3]:
def run(cls, nums1, m, nums2, n):
    nums1 = nums1[:]                 # don't mutate the shared test input
    cls().merge(nums1, m, nums2, n)
    return nums1

cases = [
    (([1, 2, 3, 0, 0, 0], 3, [2, 5, 6], 3), [1, 2, 2, 3, 5, 6]),
    (([1], 1, [], 0), [1]),
    (([0], 0, [1], 1), [1]),
    (([4, 5, 6, 0, 0, 0], 3, [1, 2, 3], 3), [1, 2, 3, 4, 5, 6]),   # all of nums2 goes first
    (([-1, 0, 0, 0], 1, [-3, -2, 5], 3), [-3, -2, -1, 5]),
]
for args, expected in cases:
    assert run(SolutionBrute, *args) == expected
    assert run(Solution, *args) == expected
    print(args, "->", expected)
print("All tests passed ✅")

([1, 2, 3, 0, 0, 0], 3, [2, 5, 6], 3) -> [1, 2, 2, 3, 5, 6]
([1], 1, [], 0) -> [1]
([0], 0, [1], 1) -> [1]
([4, 5, 6, 0, 0, 0], 3, [1, 2, 3], 3) -> [1, 2, 3, 4, 5, 6]
([-1, 0, 0, 0], 1, [-3, -2, 5], 3) -> [-3, -2, -1, 5]
All tests passed ✅


### Step 4 · Wrap up


| | Time | Space | Note |
|---|---|---|---|
| Brute force | O((m+n) log(m+n)) | O(1)* | ignores existing order |
| Optimized | O(m+n) | O(1) | one backward pass |

**Edge cases covered:** `n = 0`, `m = 0`, all of `nums2` smaller than `nums1`, negatives and ties.

**Likely follow-ups**
- *Why loop on `j` and not on `i`?* Leftover `nums1` elements are already in their final spots, but leftover `nums2` elements still have to be copied.
- *What if `nums1` didn't have spare room?* Allocate an output array and do a front-to-back merge in O(m+n) time and O(m+n) space.

🗣️ **30-second recap:** *"The brute force copies and sorts in O((m+n) log(m+n)). Since both inputs are sorted, I merge instead. To do it in place without overwriting unread values, I fill `nums1` from the back with the larger of the two current tails. That's one pass: O(m+n) time and O(1) extra space."*